# 13-bob. 13-BOB. ADVERSARIAL MASHINALI O'QITISH
## Amaliy misol: qo'shish orqali chetlab o'tish va unga qarshi himoya

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/13-bob/13-1_qoshish-orqali-chetlab-otish-va.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 13-bob.*

Quyidagi dasturda 7-bobdagiga o'xshash statik detektor sun'iy ma'lumotlarda quriladi. Har bir fayl 200 ta ikkilik belgi bilan tavsiflanadi: ma'lum funksiya import qilinganmi yoki ma'lum matnli satr bormi. Ba'zi belgilar zararli, ba'zilari esa zararsiz fayllarda ko'proq uchraydi. Buzg'unchi faylga faqat belgi qo'sha oladi, olib tashlay olmaydi, aks holda dastur ishlamay qolishi mumkin. U modelning og'irliklarini biladi (oq quti) va har qadamda modelni eng ko'p "tinchlantiradigan" belgini qo'shadi. Uch xil model taqqoslanadi: oddiy logistik regressiya, adversarial o'qitilgan model va monoton model. Har bir model o'ziga qarshi maxsus moslashtirilgan hujum bilan sinaladi. Dastur amaliy mashg'ulotning A13 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(3)
B = 200   # fayldagi 200 ta mumkin bo'lgan belgi (import qilingan funksiya yoki matnli satr bor/yo'q)
# Har bir belgining zararli va zararsiz fayllarda uchrash ehtimoli
p_zararli = rng.beta(1, 6, B)
p_zararsiz = rng.beta(1, 6, B)
p_zararli[:30] += 0.08      # 0-29 belgilar zararli fayllarda ko'proq uchraydi
p_zararsiz[30:60] += 0.08   # 30-59 belgilar zararsiz fayllarda ko'proq uchraydi
p_zararli, p_zararsiz = p_zararli.clip(0, 1), p_zararsiz.clip(0, 1)

def fayllar(p, n):
    return (rng.random((n, B)) < p).astype(float)

X = np.vstack([fayllar(p_zararsiz, 4000), fayllar(p_zararli, 4000)])
y = np.array([0] * 4000 + [1] * 4000)
sinov_zararli = fayllar(p_zararli, 1000)

def chetlab_otish(model, X, qadam):
    """Buzg'unchi faylga faqat belgi QO'SHA oladi (dastur ishlashi buzilmasligi uchun).
    U "eng zararsiz" ko'rinadigan, faylda hali yo'q belgilarni qo'shadi."""
    X = X.copy()
    w = model.coef_[0]
    for _ in range(qadam):
        nomzod = np.where(X == 0, w, np.inf)        # faqat yo'q belgilar
        X[np.arange(len(X)), nomzod.argmin(axis=1)] = 1
    return X

def natija(nom, model):
    qatorlar = []
    for q in [0, 5, 10, 20]:
        aniqlash = model.predict(chetlab_otish(model, sinov_zararli, q)).mean()
        qatorlar.append(f"{q:>2} ta: {aniqlash:5.1%}")
    print(f"{nom:<32}" + "  ".join(qatorlar))

print("Qo'shilgan belgilar soni va aniqlangan zararli fayllar ulushi:")
oddiy = LogisticRegression(max_iter=2000).fit(X, y)
natija("Oddiy model", oddiy)

# Himoya 1: buzg'unchi o'zgartirgan fayllar bilan qayta o'qitish (adversarial o'qitish)
adv = oddiy
for q in [5, 10, 20]:                          # bir necha bosqichda: model -> hujum -> qayta o'qitish
    qoshimcha = chetlab_otish(adv, fayllar(p_zararli, 2000), q)
    X, y = np.vstack([X, qoshimcha]), np.concatenate([y, np.ones(2000)])
    adv = LogisticRegression(max_iter=2000).fit(X, y)
natija("Adversarial o'qitilgan model", adv)

# Himoya 2: monoton model - belgi qo'shilishi zararli ehtimolini kamaytira olmaydi
X, y = X[:8000], y[:8000]                      # dastlabki toza ma'lumotlar
mon = LogisticRegression(max_iter=2000).fit(X, y)
mon.coef_ = np.clip(mon.coef_, 0, None)        # manfiy og'irliklarni nolga tenglaymiz
oraliq = np.quantile(mon.decision_function(X[y == 0]), 0.99)
mon.intercept_ = mon.intercept_ - oraliq       # zararsiz fayllarning 1% i xato belgilanadigan chegara
natija("Monoton model", mon)

for nom, m in [("Oddiy", oddiy), ("Adversarial", adv), ("Monoton", mon)]:
    print(f"{nom:<12} zararsiz fayllarni xato belgilash: {m.predict(fayllar(p_zararsiz, 2000)).mean():.1%}")

Qo'shilgan belgilar soni va aniqlangan zararli fayllar ulushi:
Oddiy model                      0 ta: 100.0%   5 ta: 91.3%  10 ta: 48.6%  20 ta:  0.5%


Adversarial o'qitilgan model     0 ta: 99.9%   5 ta: 97.7%  10 ta: 88.7%  20 ta: 43.2%
Monoton model                    0 ta: 99.7%   5 ta: 99.7%  10 ta: 99.7%  20 ta: 99.7%
Oddiy        zararsiz fayllarni xato belgilash: 0.0%
Adversarial  zararsiz fayllarni xato belgilash: 0.0%
Monoton      zararsiz fayllarni xato belgilash: 1.3%


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Qo'shilgan belgilar soni va aniqlangan zararli fayllar ulushi:
Oddiy model                      0 ta: 100.0%   5 ta: 91.3%  10 ta: 48.6%  20 ta:  0.5%
Adversarial o'qitilgan model     0 ta: 99.9%   5 ta: 97.7%  10 ta: 88.7%  20 ta: 43.2%
Monoton model                    0 ta: 99.7%   5 ta: 99.7%  10 ta: 99.7%  20 ta: 99.7%
Oddiy        zararsiz fayllarni xato belgilash: 0.0%
Adversarial  zararsiz fayllarni xato belgilash: 0.0%
Monoton      zararsiz fayllarni xato belgilash: 1.3%
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 13-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.